# Entity Resolution Dataset Exploration
This notebook explores the business entity datasets, visualizes the noise patterns, and analyzes the ground truth matching structure.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', None)

### 1. Load the Datasets
We use `sep='\t'` because the data is tab-separated. The ID list columns and addresses contain commas.

In [2]:
print("Loading datasets...")
train_s1 = pd.read_csv('../../../dataset/train/train_source1.tsv', sep='\t', dtype=str).fillna('')
train_s2 = pd.read_csv('../../../dataset/train/train_source2.tsv', sep='\t', dtype=str).fillna('')
train_s3 = pd.read_csv('../../../dataset/train/train_source3.tsv', sep='\t', dtype=str).fillna('')
gt = pd.read_csv('../../../dataset/train/train_ground_truth.tsv', sep='\t', dtype=str).fillna('')

print(f"Source 1 records: {len(train_s1)}")
print(f"Source 2 records: {len(train_s2)}")
print(f"Source 3 records: {len(train_s3)}")
print(f"Ground truth labels: {len(gt)}")

Loading datasets...
Source 1 records: 2206821
Source 2 records: 5034616
Source 3 records: 5285603
Ground truth labels: 2206821


### 2. Preview the Raw Data

In [3]:
train_s1.head(3)

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


In [4]:
train_s2.head(3)

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India


### 3. Analyze Ground Truth Matches
Let's see what a positive match looks like between Source 1 and Source 2/3.

In [5]:
# Get a record that has multiple matches
gt_with_matches = gt[gt['matched_entity_ids'] != '']
sample_match = gt_with_matches.iloc[0]

s1_id = sample_match['source1_entity_id']
matched_ids = sample_match['matched_entity_ids'].split(',')

print("Reference Record (Source 1):")
display(train_s1[train_s1['entity_id'] == s1_id])

print("\nGround Truth Matches (Source 2 & 3):")
s23_combined = pd.concat([train_s2, train_s3])
display(s23_combined[s23_combined['entity_id'].isin(matched_ids)])

Reference Record (Source 1):


,entity_id,business_name,business_address,country
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US



Ground Truth Matches (Source 2 & 3):


,entity_id,business_name,business_address,country
1376739,S2-681193310,Maure Wilblims Colombier Inc,,US
2003754,S2-743505751,Maure Williams Colombier,,US
63830,S3-860443364,Maure Williams Inc Center,,US
1217885,S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US
5223525,S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",US


### 4. Analyze Singletons (No Match)
Singletons are S1 entities with no matches in S2 or S3. Correctly predicting 'no match' is highly rewarded.

In [6]:
singletons = gt[gt['matched_entity_ids'] == '']
print(f"Total Singletons: {len(singletons)} out of {len(gt)} ({(len(singletons)/len(gt))*100:.2f}%)")

sample_singleton_id = singletons.iloc[0]['source1_entity_id']
display(train_s1[train_s1['entity_id'] == sample_singleton_id])

Total Singletons: 123247 out of 2206821 (5.58%)


,entity_id,business_name,business_address,country
659874,S1-302869473,International Automation Consultants Inc,"329 Rev Walton Drive, Lockport, IL",US


### 5. Check Missing Values & Missing Address Patterns

In [7]:
empty_addresses_s2 = len(train_s2[train_s2['business_address'] == ''])
empty_addresses_s3 = len(train_s3[train_s3['business_address'] == ''])

print(f"Source 2 records missing addresses: {empty_addresses_s2}")
print(f"Source 3 records missing addresses: {empty_addresses_s3}")

Source 2 records missing addresses: 168967
Source 3 records missing addresses: 175916
